In [ ]:
"""
================================================================================
  Flood-Driven River Planform Change Detection — Indus Basin (Chenab River)
  -------------------------------------------------------------------------
  Author      : Awan, A.

  Pipeline
  --------
  STAGE 1 — Google Earth Engine
    Collection : COPERNICUS/S2_SR_HARMONIZED (SR, DN scaled by 10000)
    Cloud mask : Dual — QA60 bitmask (bits 10,11) + SCL band (classes 3,8,9,10)
                 QA60 valid pre-2022-01-25 and reconstructed post-2024-02-28;
                 SCL is always present in SR collection → dual mask covers all dates
    Index      : MNDWI = (B3 − B11) / (B3 + B11)  [Xu, 2006]
                 B3=Green (10m), B11=SWIR1 (20m, resampled by GEE to 10m)
    Threshold  : Otsu dynamic per scene — clamped to [-0.1, 0.3] to prevent
                 all-zero exports when histogram is land-dominated
    Export     : Binary uint8 GeoTIFF → Google Drive (UTM 42N / EPSG:32642)

  STAGE 2 — Local post-processing
    Planform change : erosion / deposition maps + areas (km²)
    Hydrograph      : metrics from GRDC .day files (peak Q, duration, cumulative)
    ML              : Random Forest regression + SHAP explainability
    Output          : Excel workbooks + publication-quality PNG figures


  GEE Catalog
  -----------
  Collection : COPERNICUS/S2_SR_HARMONIZED
  Bands      : B3 (Green, 10m), B11 (SWIR1, 20m), QA60 (bitmask), SCL (20m)
  Scaling    : .divide(10000) after cloud masking → reflectance [0, 1]

  References
  ----------
  Leenman, A.S. (2023). GRL, 50, e2023GL103875.
  Xu, H. (2006). Int. J. Remote Sens., 27(14), 3025–3033.
  Lundberg & Lee (2017). NeurIPS, 30.
================================================================================
"""

# ── IMPORTS ───────────────────────────────────────────────────────────────────
import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from matplotlib_scalebar.scalebar import ScaleBar
import rasterio
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
import shap
import warnings
import os
import sys
import time
import shutil

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "font.family"       : "Arial",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "figure.dpi"        : 150,
})

# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

STUDY_REACH = {
    "name"    : "Chenab River — Trimmu Reach, Punjab, Pakistan",
    "lon_min" : 72.10,
    "lat_min" : 30.95,
    "lon_max" : 72.60,
    "lat_max" : 31.25,
}

# Post windows widened and cloud filter relaxed — Punjab monsoon season
# has near-continuous cloud cover; 20% filter left composites empty.
# Pre windows moved to drier months (May-Jun, May-Jul) for clean baselines.
EVENTS = {
    "2022_monsoon": {
        "label"         : "2022 Pakistan Megaflood",
        "pre_window"    : ("2022-05-01", "2022-06-30"),   # pre-monsoon baseline
        "post_window"   : ("2022-09-15", "2022-11-15"),   # post-peak recession
        "discharge_file": "discharge_2022.day",
    },
    "2014_flood": {
        "label"         : "2014 Chenab Flood",
        "pre_window"    : ("2014-06-01", "2014-08-01"),
        "post_window"   : ("2014-09-20", "2014-11-15"),
        "discharge_file": "discharge_2014.day",
    },
    "2023_flood": {
        "label"         : "2023 Post-Monsoon Event",
        "pre_window"    : ("2023-05-01", "2023-07-01"),
        "post_window"   : ("2023-08-25", "2023-10-31"),
        "discharge_file": "discharge_2023.day",
    },
}

GEE_PROJECT_ID    = "your-project-id"   # ← your GEE project ID
GEE_EXPORT_FOLDER = "GEE_RiverMobility_Chenab"
SCALE_M           = 10          # export resolution (m) — Sentinel-2 native
MAX_PIXELS        = 1e10
CRS_EXPORT        = "EPSG:32642"  # UTM Zone 42N
CLOUD_FILTER_PCT  = 40           # relaxed from 20 → monsoon cloud cover

# Otsu threshold clamp — prevents all-zero exports on land-dominated scenes
OTSU_MIN = -0.1   # below this → no water signal → clamp up
OTSU_MAX =  0.3   # above this → Otsu likely failed → clamp down

DRIVE_ROOT      = "/content/drive/MyDrive"
LOCAL_MASKS_DIR = "data/masks"
DISCHARGE_DIR   = "data/discharge_data"
OUTPUT_DIR      = "outputs"

for d in [LOCAL_MASKS_DIR, DISCHARGE_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 1 — GOOGLE EARTH ENGINE
# ══════════════════════════════════════════════════════════════════════════════

def mask_s2_clouds(image: ee.Image) -> ee.Image:
    """
    Dual cloud masking for COPERNICUS/S2_SR_HARMONIZED.

    1) QA60 bitmask — bits 10 (opaque cloud) and 11 (cirrus):
       Valid pre-2022-01-25; absent 2022-01-25 to 2024-02-28;
       reconstructed from MSK_CLASSI post-2024-02-28.
       Applied regardless — harmless when missing.

    2) SCL (Scene Classification Layer, 20m) — always present in SR:
       class 3  = cloud shadow
       class 8  = medium-probability cloud
       class 9  = high-probability cloud
       class 10 = thin cirrus

    Divides by 10000 to convert DN → surface reflectance [0, 1].
    """
    qa = image.select("QA60")
    qa_mask = (
        qa.bitwiseAnd(1 << 10).eq(0)
          .And(qa.bitwiseAnd(1 << 11).eq(0))
    )

    scl = image.select("SCL")
    scl_mask = (
        scl.neq(3)
           .And(scl.neq(8))
           .And(scl.neq(9))
           .And(scl.neq(10))
    )

    return image.updateMask(qa_mask.And(scl_mask)).divide(10000)


def compute_mndwi(image: ee.Image) -> ee.Image:
    """
    MNDWI = (Green − SWIR1) / (Green + SWIR1)   [Xu, 2006]
    S2 SR Harmonised: Green=B3 (10m), SWIR1=B11 (20m → resampled to 10m by GEE)
    Range: [−1, +1]; open water typically > 0.
    """
    return image.addBands(
        image.normalizedDifference(["B3", "B11"]).rename("MNDWI")
    )


def otsu_threshold(mndwi_img: ee.Image, aoi: ee.Geometry) -> ee.Number:
    """
    Compute Otsu optimal threshold on MNDWI histogram within AOI.
    Result is clamped to [OTSU_MIN, OTSU_MAX] to prevent degenerate
    thresholds on land-dominated or cloud-heavy composites.
    """
    histogram = mndwi_img.select("MNDWI").reduceRegion(
        reducer   = ee.Reducer.histogram(255, 0.01),
        geometry  = aoi,
        scale     = SCALE_M,
        maxPixels = 1e9,
    ).get("MNDWI")

    counts    = ee.Array(ee.Dictionary(histogram).get("histogram"))
    means     = ee.Array(ee.Dictionary(histogram).get("bucketMeans"))
    size      = means.length().get([0])
    total     = counts.reduce(ee.Reducer.sum(), [0]).get([0])
    sum_total = means.multiply(counts).reduce(ee.Reducer.sum(), [0]).get([0])

    def _iter(current, prev):
        prev  = ee.Dictionary(prev)
        idx   = ee.Number(current)
        w0    = ee.Number(prev.get("w0")).add(counts.get([idx]).divide(total))
        w1    = ee.Number(1).subtract(w0)
        sum0  = ee.Number(prev.get("sum0")).add(
                    means.get([idx]).multiply(counts.get([idx]).divide(total)))
        mean0 = sum0.divide(w0)
        mean1 = ee.Number(sum_total).subtract(sum0).divide(w1)
        var_  = w0.multiply(w1).multiply(mean0.subtract(mean1).pow(2))
        return ee.Dictionary({
            "w0"       : w0,
            "sum0"     : sum0,
            "variance" : var_,
            "threshold": ee.Algorithms.If(
                var_.gt(ee.Number(prev.get("variance"))),
                means.get([idx]),
                prev.get("threshold"),
            ),
        })

    result = ee.List.sequence(0, size.subtract(1)).iterate(
        _iter, ee.Dictionary({"w0": 0, "sum0": 0, "variance": 0, "threshold": 0})
    )
    raw = ee.Number(ee.Dictionary(result).get("threshold"))

    # Clamp to physically meaningful MNDWI range for rivers
    return raw.max(ee.Number(OTSU_MIN)).min(ee.Number(OTSU_MAX))


def build_water_mask(
    date_start : str,
    date_end   : str,
    aoi        : ee.Geometry,
    label      : str,
) -> "ee.Image | None":
    """
    Build a binary water mask (uint8: 1=water, 0=land) for a date window.

    Steps:
      1. Filter S2_SR_HARMONIZED by AOI, date, cloud cover < CLOUD_FILTER_PCT
      2. Apply dual QA60+SCL cloud mask + divide by 10000
      3. Compute MNDWI for each scene
      4. Median composite → clip to AOI
      5. Compute clamped Otsu threshold
      6. Apply threshold → binary mask as uint8

    Note: .toUint8() used directly (no selfMask/unmask roundtrip) to avoid
    all-zero exports when threshold is near the data maximum.

    Returns None if no scenes pass the cloud filter.
    """
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(date_start, date_end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUD_FILTER_PCT))
        .map(mask_s2_clouds)
        .map(compute_mndwi)
    )

    n_scenes = collection.size().getInfo()
    print(f"  [{label}]  {n_scenes} scenes  ({date_start} → {date_end}, "
          f"cloud<{CLOUD_FILTER_PCT}%)")

    if n_scenes == 0:
        print(f"  [{label}]  WARNING: 0 scenes — try widening date window "
              f"or increasing CLOUD_FILTER_PCT")
        return None

    # Median MNDWI composite
    mndwi_composite = collection.select("MNDWI").median().clip(aoi)

    # Debug: print MNDWI stats to confirm water signal exists
    stats = mndwi_composite.reduceRegion(
        reducer   = ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
        geometry  = aoi,
        scale     = 100,          # coarse scale for speed
        maxPixels = 1e8,
    ).getInfo()
    print(f"  [{label}]  MNDWI stats → "
          f"min={stats.get('MNDWI_min', 'N/A'):.3f}  "
          f"mean={stats.get('MNDWI_mean', 'N/A'):.3f}  "
          f"max={stats.get('MNDWI_max', 'N/A'):.3f}")

    threshold = otsu_threshold(mndwi_composite, aoi)
    thresh_val = threshold.getInfo()
    print(f"  [{label}]  Otsu threshold (clamped) = {thresh_val:.4f}")

    # Binary mask — direct toUint8(), no selfMask/unmask
    water = (
        mndwi_composite
        .gt(threshold)
        .rename(label)
        .toUint8()
    )
    return water


def export_mask_to_drive(
    image    : ee.Image,
    aoi      : ee.Geometry,
    filename : str,
) -> ee.batch.Task:
    """Export binary water mask as GeoTIFF to Google Drive."""
    task = ee.batch.Export.image.toDrive(
        image          = image,
        description    = filename,
        folder         = GEE_EXPORT_FOLDER,
        fileNamePrefix = filename,
        region         = aoi,
        scale          = SCALE_M,
        crs            = CRS_EXPORT,
        maxPixels      = MAX_PIXELS,
    )
    task.start()
    print(f"  Export started → Drive/{GEE_EXPORT_FOLDER}/{filename}.tif")
    return task


def wait_for_exports(tasks: list, max_wait_min: int = 60) -> None:
    """Poll GEE export tasks every 30 s until all done or timeout."""
    pending = list(tasks)
    start   = time.time()
    print(f"\n[GEE] Waiting for {len(pending)} export tasks ...")
    while pending:
        time.sleep(30)
        pending = [t for t in pending
                   if t.status()["state"] not in ("COMPLETED", "FAILED", "CANCELLED")]
        elapsed = (time.time() - start) / 60
        print(f"  {len(pending)} remaining  ({elapsed:.1f} min)")
        if elapsed > max_wait_min:
            print("  Timeout — monitor at https://code.earthengine.google.com/tasks")
            break
    print("[GEE] Exports done.")


def copy_masks_from_drive() -> None:
    """Copy exported TIFFs from Google Drive to LOCAL_MASKS_DIR."""
    src = os.path.join(DRIVE_ROOT, GEE_EXPORT_FOLDER)
    if not os.path.isdir(src):
        print(f"[DRIVE] Folder not found: {src}")
        return
    n = 0
    for f in os.listdir(src):
        if f.endswith(".tif"):
            dst = os.path.join(LOCAL_MASKS_DIR, f)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(src, f), dst)
                n += 1
    print(f"[DRIVE] Copied {n} file(s) → {LOCAL_MASKS_DIR}/")


def run_gee_stage() -> None:
    """Authenticate, build water masks for all events, export to Drive."""
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)

    aoi = ee.Geometry.Rectangle([
        STUDY_REACH["lon_min"], STUDY_REACH["lat_min"],
        STUDY_REACH["lon_max"], STUDY_REACH["lat_max"],
    ])

    tasks = []
    for name, ev in EVENTS.items():
        print(f"\n[GEE] ── {name} ──────────────────────────────────────")
        pre  = build_water_mask(*ev["pre_window"],  aoi, f"{name}_pre")
        post = build_water_mask(*ev["post_window"], aoi, f"{name}_post")
        if pre is None or post is None:
            print(f"  Skipping {name} — insufficient imagery in one window.")
            continue
        tasks.append(export_mask_to_drive(pre,  aoi, f"water_{name}_pre"))
        tasks.append(export_mask_to_drive(post, aoi, f"water_{name}_post"))

    if tasks:
        wait_for_exports(tasks)
        copy_masks_from_drive()
    print("\n[GEE] Stage 1 complete.")


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 2A — PLANFORM CHANGE DETECTION
# ══════════════════════════════════════════════════════════════════════════════

def load_geotiff(filepath: str) -> tuple:
    """Load GeoTIFF → (uint8 array, rasterio profile). Prints unique values."""
    with rasterio.open(filepath) as src:
        arr     = src.read(1).astype(np.uint8)
        profile = src.profile
    uniq = np.unique(arr)
    pct_water = 100 * arr.sum() / arr.size
    print(f"  Loaded: {os.path.basename(filepath):<40}  "
          f"shape={arr.shape}  unique={uniq}  water={pct_water:.1f}%")
    if arr.max() == 0:
        print(f"  !! WARNING: mask is all zeros — check GEE export / Otsu threshold")
    return arr, profile


def compute_planform_change(
    pre  : np.ndarray,
    post : np.ndarray,
    pixel_area_m2: float = SCALE_M ** 2,
) -> dict:
    """
    Pixel-wise planform change classification:
      Erosion (bank retreat)    : pre=0, post=1
      Deposition (bar accretion): pre=1, post=0
      Persistent water          : pre=1, post=1
      Persistent land           : pre=0, post=0
    """
    erosion    = ((pre == 0) & (post == 1)).astype(np.uint8)
    deposition = ((pre == 1) & (post == 0)).astype(np.uint8)
    persistent = ((pre == 1) & (post == 1)).astype(np.uint8)

    def km2(px): return px * pixel_area_m2 / 1e6

    return {
        "erosion_km2"    : km2(int(erosion.sum())),
        "deposition_km2" : km2(int(deposition.sum())),
        "net_change_km2" : km2(int(erosion.sum()) - int(deposition.sum())),
        "erosion_map"    : erosion,
        "deposition_map" : deposition,
        "persistent_map" : persistent,
    }


def plot_change_map(
    pre        : np.ndarray,
    post       : np.ndarray,
    change     : dict,
    profile    : dict,
    event_label: str,
    save_path  : str,
) -> None:
    """Three-panel planform change figure with scalebar, north arrow, legend."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 6))
    fig.suptitle(
        f"Flood-Driven River Planform Change — {event_label}\n{STUDY_REACH['name']}",
        fontsize=11, fontweight="bold", y=1.02,
    )

    # Change layer: 0=land, 1=erosion, 2=deposition, 3=persistent water
    ch = np.zeros_like(pre, dtype=np.uint8)
    ch[change["erosion_map"]    == 1] = 1
    ch[change["deposition_map"] == 1] = 2
    ch[change["persistent_map"] == 1] = 3

    datasets = [pre, post, ch]
    cmaps    = [
        ListedColormap(["#f5f0e8", "#3a86ff"]),
        ListedColormap(["#f5f0e8", "#3a86ff"]),
        ListedColormap(["#f5f0e8", "#e63946", "#457b9d", "#1d3557"]),
    ]
    vmaxes = [1, 1, 3]
    titles = ["(a) Pre-flood", "(b) Post-flood", "(c) Planform change"]

    for ax, data, cmap, vmax, title in zip(axes, datasets, cmaps, vmaxes, titles):
        ax.imshow(data, cmap=cmap, vmin=0, vmax=vmax, interpolation="nearest")
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.axis("off")

    # Scalebar
    try:
        t   = profile.get("transform")
        res = abs(t.a) if hasattr(t, "a") else abs(t[0])
    except Exception:
        res = SCALE_M
    axes[2].add_artist(ScaleBar(
        res, "m", length_fraction=0.25,
        location="lower right", font_properties={"size": 8}
    ))

    # North arrow
    axes[2].annotate("N", xy=(0.05, 0.96), xycoords="axes fraction",
                     fontsize=13, fontweight="bold", ha="center")
    axes[2].annotate("", xy=(0.05, 0.89), xytext=(0.05, 0.96),
                     xycoords="axes fraction",
                     arrowprops=dict(arrowstyle="->", lw=2.0, color="black"))

    # Legend
    axes[2].legend(handles=[
        mpatches.Patch(color="#e63946",
                       label=f"Lateral erosion  ({change['erosion_km2']:.3f} km²)"),
        mpatches.Patch(color="#457b9d",
                       label=f"Deposition        ({change['deposition_km2']:.3f} km²)"),
        mpatches.Patch(color="#1d3557", label="Persistent water"),
        mpatches.Patch(color="#f5f0e8", label="Persistent land",
                       edgecolor="gray", linewidth=0.5),
    ], loc="upper right", fontsize=8, framealpha=0.9)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {save_path}")


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 2B — DISCHARGE + HYDROGRAPH METRICS
# ══════════════════════════════════════════════════════════════════════════════

def load_grdc_discharge(filepath: str) -> "pd.Series | None":
    """
    Load GRDC .day format discharge file.
    Header lines start with '#' and are skipped.
    Data rows: YYYY-MM-DD   Q(m³/s)
    Missing values (-999) → NaN.
    """
    if not os.path.exists(filepath):
        print(f"  [Q] Not found: {filepath}")
        return None
    df = pd.read_csv(
        filepath,
        comment    = "#",
        sep        = r"\s+",
        header     = None,
        names      = ["date", "discharge_m3s"],
        parse_dates= ["date"],
        index_col  = "date",
    ).replace(-999.0, np.nan)
    s = df["discharge_m3s"].dropna()
    print(f"  [Q] {os.path.basename(filepath)}: "
          f"{len(s)} days, peak={s.max():,.0f} m³/s, "
          f"mean={s.mean():,.0f} m³/s")
    return s


def compute_hydrograph_metrics(series: pd.Series) -> dict:
    """
    Hydrograph metrics following Leenman (2023):
      Q_peak_m3s       : peak discharge (m³/s)
      Q_mean_m3s       : mean Q above bankfull (m³/s)
      duration_days    : days above bankfull threshold (75th percentile)
      cumulative_m3s_d : sum of Q above bankfull (m³/s · days)
      Q_rise_rate      : mean daily rise on rising limb (m³/s/day)
      Q_fall_rate      : mean daily fall on falling limb (m³/s/day)
      asymmetry        : rise duration / total above-bankfull duration
      bankfull_m3s     : bankfull proxy (75th percentile)
    """
    q        = series.dropna()
    bankfull = float(np.percentile(q.values, 75))
    above    = q[q > bankfull]
    if len(above) == 0:
        return {}

    peak_idx = q.idxmax()
    rising   = q.loc[:peak_idx]
    falling  = q.loc[peak_idx:]

    return {
        "Q_peak_m3s"       : float(q.max()),
        "Q_mean_m3s"       : float(above.mean()),
        "duration_days"    : int(len(above)),
        "cumulative_m3s_d" : float(above.values.sum()),
        "Q_rise_rate"      : float(rising.diff().clip(lower=0).mean())
                             if len(rising) > 1 else 0.0,
        "Q_fall_rate"      : float(falling.diff().clip(upper=0).abs().mean())
                             if len(falling) > 1 else 0.0,
        "asymmetry"        : float(len(rising)) / max(len(above), 1),
        "bankfull_m3s"     : bankfull,
    }


def plot_hydrograph(
    series      : pd.Series,
    bankfull    : float,
    event_label : str,
    pre_window  : tuple,
    post_window : tuple,
    save_path   : str,
) -> None:
    """Hydrograph with bankfull line, pre/post windows shaded."""
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(series.index, series.values, color="#2E75B6", lw=1.4, label="Daily Q")
    ax.fill_between(
        series.index, series.values, bankfull,
        where=(series.values > bankfull),
        color="#e63946", alpha=0.30, label="Above bankfull"
    )
    ax.axhline(bankfull, color="gray", ls="--", lw=1,
               label=f"Bankfull ≈ {bankfull:,.0f} m³/s")

    for window, colour, lbl in [
        (pre_window,  "#2a9d8f", "Pre-flood window"),
        (post_window, "#e9c46a", "Post-flood window"),
    ]:
        ax.axvspan(pd.Timestamp(window[0]), pd.Timestamp(window[1]),
                   alpha=0.18, color=colour, label=lbl)

    ax.set_title(
        f"Discharge Hydrograph — {event_label}\n{STUDY_REACH['name']}",
        fontsize=10, fontweight="bold"
    )
    ax.set_ylabel("Q (m³/s)", fontsize=9)
    ax.set_xlabel("Date", fontsize=9)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.legend(fontsize=8, loc="upper right")
    ax.tick_params(labelsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {save_path}")


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 2C — RANDOM FOREST + SHAP
# ══════════════════════════════════════════════════════════════════════════════

FEATURE_COLS = [
    "Q_peak_m3s",
    "Q_mean_m3s",
    "duration_days",
    "cumulative_m3s_d",
    "Q_rise_rate",
    "Q_fall_rate",
    "asymmetry",
]


def train_rf_model(df: pd.DataFrame) -> tuple:
    """
    Train Random Forest: hydrograph metrics → erosion_km2.
    Leave-One-Out CV appropriate for small-N hydrological datasets.
    """
    df_c = df.dropna(subset=FEATURE_COLS + ["erosion_km2"])
    if len(df_c) < 3:
        print(f"[RF] Only {len(df_c)} complete events — need ≥3. Skipping ML.")
        return None, None, None, None, None

    X        = df_c[FEATURE_COLS].values
    y        = df_c["erosion_km2"].values
    scaler   = StandardScaler()
    X_sc     = scaler.fit_transform(X)

    rf = RandomForestRegressor(
        n_estimators     = 500,
        max_features     = "sqrt",
        min_samples_leaf = 1,
        random_state     = 42,
        n_jobs           = -1,
    )

    loo    = LeaveOneOut()
    cv_r2  = cross_val_score(rf, X_sc, y, cv=loo, scoring="r2")
    cv_mae = cross_val_score(rf, X_sc, y, cv=loo,
                             scoring="neg_mean_absolute_error")
    rf.fit(X_sc, y)

    print(f"\n[RF] Leave-One-Out CV  (n={len(y)} events)")
    print(f"     LOO R²  : {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")
    print(f"     LOO MAE : {-cv_mae.mean():.4f} km²")
    print(f"     Train R²: {r2_score(y, rf.predict(X_sc)):.3f}")

    return rf, scaler, X_sc, y, cv_r2


def plot_shap(
    model        : RandomForestRegressor,
    X_scaled     : np.ndarray,
    save_beeswarm: str,
    save_bar     : str,
) -> pd.DataFrame:
    """SHAP TreeExplainer — beeswarm + ranked bar chart + CSV."""
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_scaled)

    # Beeswarm
    shap.summary_plot(shap_values, X_scaled, feature_names=FEATURE_COLS,
                      show=False, plot_size=(8, 5))
    plt.title("SHAP Variable Importance — Flood-Driven Lateral Erosion\n"
              "Chenab River, Indus Basin", fontsize=10, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_beeswarm, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  SHAP beeswarm → {save_beeswarm}")

    # Bar chart
    mean_shap = np.abs(shap_values).mean(axis=0)
    imp = pd.DataFrame({"Feature": FEATURE_COLS, "Mean_SHAP": mean_shap})
    imp = imp.sort_values("Mean_SHAP", ascending=True)

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.barh(imp["Feature"], imp["Mean_SHAP"],
                   color="#2E75B6", edgecolor="white", height=0.6)
    ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=8)
    ax.set_xlabel("Mean |SHAP value|  (km²)", fontsize=9)
    ax.set_title("Predictor Importance — RF + SHAP\n"
                 "Target: lateral erosion area (km²)",
                 fontsize=10, fontweight="bold")
    ax.tick_params(labelsize=9)
    plt.tight_layout()
    plt.savefig(save_bar, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  SHAP bar chart → {save_bar}")

    out = imp.sort_values("Mean_SHAP", ascending=False)
    csv = os.path.join(OUTPUT_DIR, "shap_importance.csv")
    out.to_csv(csv, index=False)
    print(f"  SHAP CSV       → {csv}")
    return out


# ══════════════════════════════════════════════════════════════════════════════
#  EXCEL EXPORT
# ══════════════════════════════════════════════════════════════════════════════

def export_excel(df_summary: pd.DataFrame, discharge_dict: dict) -> None:
    """Export event summary + discharge time series to Excel."""
    # Event summary + metadata
    path1 = os.path.join(OUTPUT_DIR, "event_results.xlsx")
    with pd.ExcelWriter(path1, engine="openpyxl") as w:
        df_summary.to_excel(w, sheet_name="Event_Summary", index=False)
        pd.DataFrame({
            "Parameter": [
                "Study reach", "CRS", "Collection",
                "Cloud mask", "Cloud filter", "Water index",
                "Otsu clamp", "Scale (m)",
            ],
            "Value": [
                STUDY_REACH["name"], CRS_EXPORT,
                "COPERNICUS/S2_SR_HARMONIZED",
                "QA60 (bits 10,11) + SCL (classes 3,8,9,10)",
                f"CLOUDY_PIXEL_PERCENTAGE < {CLOUD_FILTER_PCT}%",
                "MNDWI = (B3−B11)/(B3+B11) [Xu 2006], Otsu dynamic threshold",
                f"[{OTSU_MIN}, {OTSU_MAX}]",
                SCALE_M,
            ],
        }).to_excel(w, sheet_name="Metadata", index=False)
    print(f"\n[EXCEL] Event summary → {path1}")

    # Discharge time series
    path2 = os.path.join(OUTPUT_DIR, "discharge_data.xlsx")
    with pd.ExcelWriter(path2, engine="openpyxl") as w:
        for name, s in discharge_dict.items():
            df_q = s.reset_index()
            df_q.columns = ["Date", "Discharge_m3s"]
            df_q.to_excel(w, sheet_name=name[:31], index=False)
    print(f"[EXCEL] Discharge data → {path2}")


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    print("=" * 72)
    print("  River Planform Change — Chenab River, Indus Basin")
    print(f"  Reach : {STUDY_REACH['name']}")
    print(f"  Events: {', '.join(EVENTS.keys())}")
    print("=" * 72)

    # Mount Drive (Colab)
    if "google.colab" in sys.modules:
        from google.colab import drive
        drive.mount("/content/drive")

    # ── STAGE 1: GEE ──────────────────────────────────────────────────────────
    masks_ok = {
        name: (
            os.path.exists(os.path.join(LOCAL_MASKS_DIR, f"water_{name}_pre.tif"))
            and
            os.path.exists(os.path.join(LOCAL_MASKS_DIR, f"water_{name}_post.tif"))
        )
        for name in EVENTS
    }

    if not all(masks_ok.values()):
        missing = [k for k, v in masks_ok.items() if not v]
        print(f"\n[GEE] Missing masks: {missing} — running GEE export...")
        run_gee_stage()
    else:
        print("\n[LOCAL] All masks found — skipping GEE stage.")

    # ── STAGE 2: local analysis ────────────────────────────────────────────────
    results        = []
    discharge_dict = {}

    for name, ev in EVENTS.items():
        print(f"\n{'─'*60}")
        print(f"  {ev['label']}  ({name})")
        print(f"{'─'*60}")

        pre_path  = os.path.join(LOCAL_MASKS_DIR, f"water_{name}_pre.tif")
        post_path = os.path.join(LOCAL_MASKS_DIR, f"water_{name}_post.tif")

        if not (os.path.exists(pre_path) and os.path.exists(post_path)):
            print(f"  Masks missing — skipping {name}.")
            continue

        # 2A — planform change
        pre,  pre_prof  = load_geotiff(pre_path)
        post, _         = load_geotiff(post_path)
        ch = compute_planform_change(pre, post, SCALE_M ** 2)
        print(f"  Erosion    : {ch['erosion_km2']:.4f} km²")
        print(f"  Deposition : {ch['deposition_km2']:.4f} km²")
        print(f"  Net change : {ch['net_change_km2']:.4f} km²")

        plot_change_map(
            pre, post, ch, pre_prof, ev["label"],
            os.path.join(OUTPUT_DIR, f"change_map_{name}.png"),
        )

        # 2B — hydrograph
        q_series = load_grdc_discharge(
            os.path.join(DISCHARGE_DIR, ev["discharge_file"])
        )
        metrics = {}
        if q_series is not None:
            metrics = compute_hydrograph_metrics(q_series)
            discharge_dict[name] = q_series
            plot_hydrograph(
                q_series,
                metrics.get("bankfull_m3s", float(np.percentile(q_series, 75))),
                ev["label"],
                ev["pre_window"],
                ev["post_window"],
                os.path.join(OUTPUT_DIR, f"hydrograph_{name}.png"),
            )
            print(f"  Peak Q     : {metrics['Q_peak_m3s']:,.0f} m³/s")
            print(f"  Duration   : {metrics['duration_days']} days > bankfull")
            print(f"  Cumulative : {metrics['cumulative_m3s_d']:,.0f} m³/s·days")
        else:
            print(f"  No discharge file — hydrograph metrics skipped.")

        results.append({
            "event"         : name,
            "label"         : ev["label"],
            "erosion_km2"   : ch["erosion_km2"],
            "deposition_km2": ch["deposition_km2"],
            "net_change_km2": ch["net_change_km2"],
            **metrics,
        })

    # ── STAGE 2C: RF + SHAP ───────────────────────────────────────────────────
    df = pd.DataFrame(results)
    rf, scaler, X_sc, y, cv = train_rf_model(df)
    if rf is not None:
        plot_shap(
            rf, X_sc,
            save_beeswarm = os.path.join(OUTPUT_DIR, "shap_beeswarm.png"),
            save_bar      = os.path.join(OUTPUT_DIR, "shap_bar.png"),
        )

    # ── Excel ─────────────────────────────────────────────────────────────────
    export_excel(df, discharge_dict)

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*72}")
    print("  COMPLETE — outputs/")
    for f in sorted(os.listdir(OUTPUT_DIR)):
        sz = os.path.getsize(os.path.join(OUTPUT_DIR, f))
        print(f"    {f:<45} {sz:>10,} bytes")
    print("=" * 72)


if __name__ == "__main__":
    main()